<a href="https://colab.research.google.com/github/D2718281828nis/Neuro-Neuron_Models/blob/master/assetes/Simple-Neuron-Ensamble-Synchronization-Kuramoto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Interactive SNN Synchronization: Kuramoto Oscillators

In [4]:


import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. FIXED SIMULATION SETUP
# ==========================================
np.random.seed(42)
N = 100               # Number of neurons
dt = 0.01             # Time step (ms)
T = 20.0              # Total simulation time (ms)
time = np.arange(0, T, dt)
n_steps = len(time)

# Heterogeneous natural frequencies & initial phases (fixed across K changes)
omega = np.random.normal(loc=2.0, scale=0.4, size=N)
theta_init = np.random.uniform(0, 2 * np.pi, N)

# ==========================================
# 2. SIMULATION & PLOTTING FUNCTION
# ==========================================
def update_simulation(K=1.2):
    theta = theta_init.copy()
    phase_hist = np.zeros((N, n_steps))
    spike_times = []
    order_param = np.zeros(n_steps)

    # Kuramoto integration
    for i, t in enumerate(time):
        # FIXED: Correct Kuramoto coupling -> sum_j sin(θ_j - θ_i)
        # Broadcasting: (1, N) - (N, 1) -> (N, N) matrix of [θ_j - θ_i]
        coupling = (K / N) * np.sum(np.sin(theta[np.newaxis, :] - theta[:, np.newaxis]), axis=1)

        theta += (omega + coupling) * dt

        # Store wrapped phase for heatmap [0, 2π)
        phase_hist[:, i] = theta % (2 * np.pi)

        # Global synchronization metric
        order_param[i] = np.abs(np.mean(np.exp(1j * theta)))

        # SPIKE DETECTION & RESET (LIF threshold crossing)
        spikes = theta >= 2 * np.pi
        if np.any(spikes):
            spike_idx = np.where(spikes)[0]
            spike_times.extend([[idx, t] for idx in spike_idx])
            theta[spikes] -= 2 * np.pi  # Phase reset = Voltage reset

    spike_times = np.array(spike_times) if spike_times else np.empty((0, 2))

    # ==========================================
    # 3. VISUALIZATION (Neuromorphic-Style)
    # ==========================================
    fig, axs = plt.subplots(3, 1, figsize=(11, 10), gridspec_kw={'hspace': 0.35})

    # Plot 1: Phase Heatmap (Replaces messy line plot)
    im = axs[0].imshow(phase_hist, aspect='auto', cmap='twilight', vmin=0, vmax=2*np.pi,
                       extent=[0, T, N-0.5, -0.5], origin='lower')
    axs[0].set_title(f'Neuron Phase Evolution (K = {K:.2f})', fontsize=13, fontweight='bold')
    axs[0].set_ylabel('Neuron Index')
    axs[0].set_xticks([])
    fig.colorbar(im, ax=axs[0], label='Phase θ (rad)', fraction=0.05, pad=0.02)
    axs[0].grid(False)

    # Plot 2: Spike Raster Plot (AER Event Stream)
    if len(spike_times) > 0:
        axs[1].scatter(spike_times[:, 1], spike_times[:, 0], s=2, c='black', alpha=0.7)
    axs[1].set_title('Spike Raster Plot (Asynchronous AER Communication)', fontsize=13, fontweight='bold')
    axs[1].set_ylabel('Neuron Index')
    axs[1].set_xlim(0, T)
    axs[1].set_ylim(-0.5, N+0.5)
    axs[1].grid(True, alpha=0.3)

    # Plot 3: Order Parameter
    axs[2].plot(time, order_param, color='darkorange', lw=2.5)
    axs[2].set_title('Global Synchronization Order Parameter (r)', fontsize=13, fontweight='bold')
    axs[2].set_xlabel('Time (ms)')
    axs[2].set_ylabel('r = |⟨e^{iθ}⟩|')
    axs[2].set_ylim(0, 1.05)
    axs[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='r=0.5 threshold')
    axs[2].fill_between(time, order_param, alpha=0.2, color='darkorange')
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. INTERACTIVE WIDGET
# ==========================================
interact(
    update_simulation,
    K=FloatSlider(
        min=0.0, max=3.0, step=0.05, value=1.2,
        description='Synaptic Coupling (K):',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )
)

interactive(children=(FloatSlider(value=1.2, description='Synaptic Coupling (K):', layout=Layout(width='60%'),…

<function __main__.update_simulation(K=1.2)>